# Análisis de matrices de atención en un Transformer encoder-only (BERT)

**Corpus:** *La Metamorfosis*, Franz Kafka — dos páginas consecutivas (ver `corpus_metamorfosis.txt`).

**Modelo:** `bert-base-multilingual-cased` (12 capas, 12 cabezas).

## 1. Descripción
Analizamos oraciones complejas con un modelo Transformer *encoder-only*. Inspeccionamos las matrices de atención
y estudiamos cómo distintas capas y cabezas distribuyen la atención entre tokens.

> La actividad **no** afirma que los pesos de atención sean explicaciones causales completas. Busca una lectura
> crítica de las representaciones internas de un Transformer.

## 2. Objetivos
- Tokenizar texto con el tokenizer de BERT.
- Ejecutar el modelo con `output_attentions=True`.
- Identificar dimensiones de capas, cabezas y tokens.
- Extraer los tokens con mayor atención desde tokens seleccionados.
- Comparar patrones entre oraciones, capas y cabezas.
- Analizar los límites de interpretar la atención como explicación.

## 3. Entregables cubiertos por este notebook
Notebook ejecutado · corpus · evidencia de tokens y matrices · tablas de atención (CSV) · análisis escrito (sección 6).

## Instalación

In [ ]:
!pip install -q transformers torch pandas matplotlib

In [ ]:
import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

MODEL_NAME = 'bert-base-multilingual-cased'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Dispositivo:', DEVICE)

## 4. Corpus (dos páginas consecutivas)

El texto completo se guarda en `corpus_metamorfosis.txt`. Aquí lo cargamos y de él elegimos las oraciones
que analizaremos. Si te tocaron otras páginas, reemplaza el contenido del archivo o de la variable `paginas`.

In [ ]:
paginas = '''Cuando Gregorio Samsa se despertó una mañana después de un sueño intranquilo, se encontró sobre su cama convertido en un monstruoso insecto.
Estaba tumbado sobre su espalda dura, y en forma de caparazón y, al levantar un poco la cabeza, veía un vientre abombado, parduzco, dividido por partes duras en forma de arco, sobre cuya prominencia apenas podía mantenerse el cobertor, a punto ya de resbalar al suelo.
Sus muchas patas, ridículamente pequeñas en comparación con el resto de su tamaño, le vibraban desamparadas ante los ojos.
Su habitación, una auténtica habitación humana, si bien algo pequeña, permanecía tranquila entre las cuatro paredes harto conocidas.
La mirada de Gregorio se dirigió después hacia la ventana, y el tiempo lluvioso lo puso muy melancólico.
Y cuando dieron las seis y cuarto se oyó que llamaban cautelosamente a la puerta que había junto a la cabecera de su cama.'''

# Intentamos leer el archivo del corpus si existe (tiene el texto completo de las 2 páginas).
try:
    with open('corpus_metamorfosis.txt', encoding='utf-8') as f:
        print(f.read()[:1500], '...')
except FileNotFoundError:
    print('(No se encontró corpus_metamorfosis.txt; se usa el texto embebido en la variable `paginas`.)')

# Oraciones lingüísticas que analizaremos (una simple, varias complejas).
oraciones = {
    'O1_simple':   'La mirada de Gregorio se dirigió hacia la ventana.',
    'O2_compleja': 'Cuando Gregorio Samsa se despertó una mañana después de un sueño intranquilo, se encontró sobre su cama convertido en un monstruoso insecto.',
    'O3_compleja': 'Y cuando dieron las seis y cuarto se oyó que llamaban cautelosamente a la puerta que había junto a la cabecera de su cama.',
    # Par de contraste para la Parte D (palabra ambigua: "banco").
    'B1_banco_finanzas': 'El banco aprobó el préstamo.',
    'B2_banco_parque':   'Me senté en el banco del parque.',
}
for k, v in oraciones.items():
    print(f'{k:20s} | {v}')

## Parte A · Tokenización

Cargamos el tokenizer y mostramos los tokens de cada oración. Identificamos:
- Tokens especiales `[CLS]` y `[SEP]`.
- Palabras divididas en subpalabras (prefijo `##`).
- Diferencia entre **palabra lingüística** y **token del modelo**.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Vocabulario:', tokenizer.vocab_size)
print('Especiales:', tokenizer.all_special_tokens)

In [ ]:
def inspeccionar_tokens(texto):
    enc = tokenizer(texto, return_tensors='pt')
    ids = enc['input_ids'][0].tolist()
    toks = tokenizer.convert_ids_to_tokens(ids)
    word_ids = enc.word_ids()
    filas = []
    for i, (t, wid, tid) in enumerate(zip(toks, word_ids, ids)):
        filas.append({
            'idx': i,
            'token': t,
            'id': tid,
            'palabra_ling_#': wid,
            'especial': t in tokenizer.all_special_tokens,
            'subpalabra': t.startswith('##'),
        })
    return enc, toks, pd.DataFrame(filas)

for nombre, texto in oraciones.items():
    print('=' * 90)
    print(nombre, '::', texto)
    _, toks, df = inspeccionar_tokens(texto)
    print('N.º de tokens del modelo:', len(toks))
    print('Tokens:', toks)
    display(df)

In [ ]:
# Palabra lingüística  vs  tokens del modelo: cuántas palabras se parten en subpalabras.
def resumen_subpalabras(texto):
    enc = tokenizer(texto, return_tensors='pt')
    toks = tokenizer.convert_ids_to_tokens(enc['input_ids'][0])
    word_ids = enc.word_ids()
    palabras = {}
    for t, wid in zip(toks, word_ids):
        if wid is None:
            continue
        palabras.setdefault(wid, []).append(t)
    n_palabras = len(palabras)
    n_tokens_utiles = sum(len(v) for v in palabras.values())
    partidas = {tuple(v): len(v) for v in palabras.values() if len(v) > 1}
    return {
        'oracion': texto,
        'palabras_ling': n_palabras,
        'tokens_modelo(sin especiales)': n_tokens_utiles,
        'tokens_totales(con [CLS]/[SEP])': len(toks),
        'palabras_partidas': partidas,
    }

resumen = pd.DataFrame([resumen_subpalabras(t) for t in oraciones.values()])
display(resumen)

## Parte B · Ejecución del modelo

Ejecutamos el modelo pidiendo las atenciones y reportamos la forma de las matrices.
Estructura esperada por capa: `(batch_size, n_cabezas, n_tokens, n_tokens)`.

In [ ]:
modelo = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True).to(DEVICE)
modelo.eval()
print('Capas (encoder layers):', modelo.config.num_hidden_layers)
print('Cabezas por capa:', modelo.config.num_attention_heads)

In [ ]:
def correr_modelo(texto):
    entradas = tokenizer(texto, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        outputs = modelo(**entradas)
    attentions = outputs.attentions  # tuple de longitud n_capas
    toks = tokenizer.convert_ids_to_tokens(entradas['input_ids'][0])
    return entradas, attentions, toks

entradas, attentions, toks = correr_modelo(oraciones['O2_compleja'])
print('Tipo de outputs.attentions :', type(attentions))
print('N.º de elementos (= n.º de capas):', len(attentions))
print('Forma de attentions[0]:', tuple(attentions[0].shape))
print('  -> (batch_size, n_cabezas, n_tokens, n_tokens)')
print('Tokens de esta oración:', toks)

# Comprobación: cada fila de la matriz de atención suma ~1 (es una distribución softmax).
fila = attentions[5][0, 3, 1]
print('Suma de una fila de atención (capa 5, cabeza 3, token 1):', float(fila.sum()))

In [ ]:
# Reporte de formas para TODAS las oraciones.
filas = []
attn_cache = {}
for nombre, texto in oraciones.items():
    ent, att, tk = correr_modelo(texto)
    attn_cache[nombre] = (att, tk)
    filas.append({
        'oracion': nombre,
        'n_capas': len(att),
        'forma_por_capa': tuple(att[0].shape),
        'n_cabezas': att[0].shape[1],
        'n_tokens': att[0].shape[-1],
    })
display(pd.DataFrame(filas))

## Parte C · Análisis de atención

Seleccionamos:
- **2 capas distintas:** 3 y 8 (temprana vs. profunda).
- **2 cabezas distintas:** 2 y 9.
- **2 tokens lingüísticamente relevantes por oración** (verbo/sustantivo núcleo).

Para cada combinación mostramos los **5 tokens con mayor peso de atención** desde el token consultado.

In [ ]:
def indices_de_palabra(toks, palabra):
    """Devuelve la lista de índices de token que componen `palabra` (maneja subpalabras ##)."""
    objetivo = palabra.lower()
    acumulado, inicio = '', None
    for i, t in enumerate(toks):
        pieza = t[2:] if t.startswith('##') else t
        if t.startswith('##') and inicio is not None:
            acumulado += pieza
        else:
            acumulado, inicio = pieza, i
        if acumulado.lower() == objetivo:
            return list(range(inicio, i + 1))
    # Coincidencia parcial (primer token que empieza igual)
    for i, t in enumerate(toks):
        if t.lower().lstrip('#').startswith(objetivo[:4]):
            return [i]
    raise ValueError(f'No encontré "{palabra}" en {toks}')


def top_atencion(attentions, toks, capa, cabeza, idx_consulta, k=5):
    """Top-k tokens atendidos desde `idx_consulta`. Si son varios subtokens, se promedia la fila."""
    if isinstance(idx_consulta, int):
        idx_consulta = [idx_consulta]
    fila = attentions[capa][0, cabeza, idx_consulta, :].mean(dim=0)
    vals, idx = fila.topk(k)
    return [(toks[j], round(float(v), 4)) for v, j in zip(vals, idx)]


# Tokens relevantes elegidos a mano para cada oración analizada.
tokens_relevantes = {
    'O1_simple':   ['mirada', 'ventana'],
    'O2_compleja': ['despertó', 'insecto'],
    'O3_compleja': ['llamaban', 'puerta'],
    'B1_banco_finanzas': ['banco', 'préstamo'],
    'B2_banco_parque':   ['banco', 'parque'],
}

CAPAS = [3, 8]
CABEZAS = [2, 9]

registros = []
for nombre, palabras in tokens_relevantes.items():
    att, tk = attn_cache[nombre]
    for palabra in palabras:
        idxs = indices_de_palabra(tk, palabra)
        for capa in CAPAS:
            for cabeza in CABEZAS:
                top = top_atencion(att, tk, capa, cabeza, idxs, k=5)
                registros.append({
                    'oracion': nombre,
                    'token_consulta': palabra,
                    'subtokens': [tk[i] for i in idxs],
                    'capa': capa,
                    'cabeza': cabeza,
                    'top5_tokens_atendidos': [t for t, _ in top],
                    'top5_pesos': [p for _, p in top],
                })

tabla_C = pd.DataFrame(registros)
display(tabla_C)

In [ ]:
# Vista más legible: una fila por (oración, token, capa, cabeza) con el top-5 como texto.
vista = tabla_C.copy()
vista['top5'] = [
    ', '.join(f'{t}({p})' for t, p in zip(ts, ps))
    for ts, ps in zip(vista['top5_tokens_atendidos'], vista['top5_pesos'])
]
display(vista[['oracion', 'token_consulta', 'capa', 'cabeza', 'top5']])

In [ ]:
# Mapa de calor de una matriz de atención (capa/cabeza a elección) para evidencia visual.
def heatmap(nombre_oracion, capa, cabeza):
    att, tk = attn_cache[nombre_oracion]
    M = att[capa][0, cabeza].cpu().numpy()
    fig, ax = plt.subplots(figsize=(0.45 * len(tk) + 2, 0.45 * len(tk) + 2))
    im = ax.imshow(M, cmap='viridis')
    ax.set_xticks(range(len(tk))); ax.set_xticklabels(tk, rotation=90, fontsize=8)
    ax.set_yticks(range(len(tk))); ax.set_yticklabels(tk, fontsize=8)
    ax.set_xlabel('token atendido (key)'); ax.set_ylabel('token consulta (query)')
    ax.set_title(f'{nombre_oracion} — capa {capa}, cabeza {cabeza}')
    fig.colorbar(im, fraction=0.046, pad=0.04)
    plt.tight_layout(); plt.show()

heatmap('O2_compleja', 3, 2)
heatmap('O2_compleja', 8, 9)

## Parte D · Comparación entre oraciones

Contraste con una palabra ambigua: **"banco"** en dos contextos.

```
El banco aprobó el préstamo.       (institución financiera)
Me senté en el banco del parque.   (asiento)
```

Comparamos a qué atiende el token `banco` en cada oración, y también comparamos una oración simple con una compleja del corpus de Kafka.

In [ ]:
def comparar_token(palabra, oraciones_lista, capas=CAPAS, cabezas=CABEZAS, k=5):
    filas = []
    for nombre in oraciones_lista:
        att, tk = attn_cache[nombre]
        idxs = indices_de_palabra(tk, palabra)
        for capa in capas:
            for cabeza in cabezas:
                top = top_atencion(att, tk, capa, cabeza, idxs, k=k)
                filas.append({
                    'oracion': nombre,
                    'palabra': palabra,
                    'capa': capa, 'cabeza': cabeza,
                    'top5': ', '.join(f'{t}({p})' for t, p in top),
                })
    return pd.DataFrame(filas)

tabla_D_banco = comparar_token('banco', ['B1_banco_finanzas', 'B2_banco_parque'])
display(tabla_D_banco)

In [ ]:
# Simple vs compleja: el token "Gregorio" / "mirada" en O1 vs el sujeto en O2.
tabla_D_simple_vs_compleja = pd.concat([
    comparar_token('mirada', ['O1_simple']),
    comparar_token('despertó', ['O2_compleja']),
], ignore_index=True)
display(tabla_D_simple_vs_compleja)

# Distancia media a la que atiende cada cabeza (¿atención local o de largo alcance?).
def distancia_media_atencion(nombre, capa, cabeza):
    att, tk = attn_cache[nombre]
    M = att[capa][0, cabeza].cpu().numpy()
    n = M.shape[0]
    pos = np.arange(n)
    d = np.abs(pos[:, None] - pos[None, :])
    return float((M * d).sum() / M.sum())

filas = []
for nombre in ['O1_simple', 'O2_compleja', 'O3_compleja']:
    for capa in CAPAS:
        for cabeza in CABEZAS:
            filas.append({'oracion': nombre, 'capa': capa, 'cabeza': cabeza,
                          'distancia_media_atencion': round(distancia_media_atencion(nombre, capa, cabeza), 2)})
display(pd.DataFrame(filas))

In [ ]:
# Guardar tablas como evidencia (entregable).
tabla_C.to_csv('tabla_atencion_parteC.csv', index=False, encoding='utf-8')
tabla_D_banco.to_csv('tabla_atencion_parteD_banco.csv', index=False, encoding='utf-8')
tabla_D_simple_vs_compleja.to_csv('tabla_atencion_parteD_simple_vs_compleja.csv', index=False, encoding='utf-8')
resumen.to_csv('tabla_tokenizacion.csv', index=False, encoding='utf-8')
print('CSV guardados:')
print(' - tabla_tokenizacion.csv')
print(' - tabla_atencion_parteC.csv')
print(' - tabla_atencion_parteD_banco.csv')
print(' - tabla_atencion_parteD_simple_vs_compleja.csv')

## 6. Preguntas de análisis (análisis escrito)

> Las afirmaciones cuantitativas concretas (qué token gana, con qué peso) deben leerse de las tablas generadas
> arriba tras ejecutar el notebook. Abajo va la interpretación esperada y el razonamiento.

### 1. ¿Qué tokens reciben mayor atención desde cada token seleccionado?
En la mayoría de cabezas, los tokens de consulta concentran gran parte de su masa de atención en:
- **`[SEP]` y `[CLS]`**, que funcionan como *"no-op"* / sumideros de atención (attention sinks): cuando una cabeza
  no tiene nada relevante que hacer en esa posición, descarga el peso ahí.
- **Tokens adyacentes** (la palabra anterior y la siguiente), sobre todo en capas tempranas.
- **Núcleos sintácticos relacionados**: p. ej. desde el verbo (`despertó`, `llamaban`) hacia su sujeto
  (`Gregorio`, `Samsa`) o su objeto/complemento; desde un sustantivo hacia su artículo y sus modificadores
  (`ventana` ← `la`; `insecto` ← `monstruoso`, `un`).

### 2. ¿Cambian los patrones entre capas?
Sí, de forma sistemática:
- **Capas tempranas (p. ej. 3):** atención muy **local y posicional** — diagonal marcada, foco en vecinos
  inmediatos y en subpalabras del mismo término. La "distancia media de atención" calculada es baja.
- **Capas profundas (p. ej. 8):** atención más **dispersa y de largo alcance**, con relaciones entre palabras
  alejadas (concordancia, correferencia, dependencia verbo–argumento) y mayor concentración en `[SEP]`/`[CLS]`.

### 3. ¿Cambian los patrones entre cabezas?
Sí. Dentro de una misma capa, distintas cabezas se "especializan":
- una cabeza sigue el token **siguiente**, otra el **anterior**;
- alguna liga **artículo–sustantivo** o **preposición–término**;
- otra manda casi todo a `[SEP]` (cabeza "apagada" para esa entrada).
Las dos cabezas elegidas (2 y 9) muestran top-5 distintos para el mismo token de consulta.

### 4. ¿Las palabras con mayor atención son lingüísticamente relevantes?
**A veces sí, a veces no.** Hay cabezas cuyo top-5 coincide con relaciones sintácticas plausibles
(verbo→sujeto, sustantivo→adjetivo). Pero una fracción grande del peso va a tokens **sin contenido**
(`[CLS]`, `[SEP]`, signos de puntuación), que no son "relevantes" en sentido lingüístico. Por eso no se puede
leer la fila de atención como si fuera un análisis sintáctico.

### 5. ¿Qué diferencias aparecen entre oraciones simples y complejas?
- En la **oración simple** (`O1`) la atención es más concentrada y la estructura "cabe" en dependencias cortas.
- En las **complejas** (`O2`, `O3`) con subordinadas y varios sintagmas, la atención se **reparte más** y
  aumentan los enlaces de larga distancia (el verbo principal atendiendo a un argumento separado por la
  subordinada). La "distancia media de atención" crece con la longitud y la complejidad.

### 6. ¿Qué ocurre cuando una palabra se divide en subpalabras?
- El modelo ve **varios tokens** donde el hablante ve una palabra (p. ej. `parduzco` → `par ##du ##zco`).
- Los subtokens de una misma palabra suelen atenderse **fuertemente entre sí** (bloque en la diagonal).
- La atención "de la palabra" no está en un solo vector: hay que **agregar** las filas de sus subtokens
  (aquí promediamos). El primer subtoken suele cargar con la representación sintáctica del conjunto.
- Complica la comparación palabra↔palabra entre oraciones, porque el número de tokens cambia.

### 7. ¿Qué NO puedes concluir observando únicamente los pesos de atención?
- **Causalidad:** que un token reciba peso alto no prueba que *cause* la predicción o la representación final.
- **Importancia real:** los pesos ignoran la magnitud de los *value vectors* y las transformaciones posteriores
  (residuales, MLP, LayerNorm, y las 11 capas restantes que mezclan todo).
- **Explicación única:** existen distribuciones de atención distintas que producen la misma salida
  (la atención no es identificable).
- **Semántica / desambiguación:** aunque `banco` cambie su patrón entre oraciones, la atención no nos dice
  *qué sentido* eligió el modelo; eso vive en los estados ocultos, no en los pesos.
- **Sintaxis formal:** coincidencias con relaciones de dependencia son parciales y dependientes de la cabeza;
  no hay garantía de un árbol coherente.
- En resumen: la atención es una **pista sobre el flujo de información**, no una explicación completa ni fiel.

## 7. Entregables — checklist

| Entregable | Dónde |
|---|---|
| Notebook `.ipynb` ejecutado | este archivo (ejecutar todo: *Runtime → Run all*) |
| Corpus (páginas seleccionadas) | `corpus_metamorfosis.txt` |
| Evidencia de tokens y matrices | Parte A (tablas de tokens) + Parte B (formas + heatmaps) |
| Tablas con resultados de atención | `tabla_atencion_parteC.csv`, `tabla_atencion_parteD_banco.csv`, `tabla_atencion_parteD_simple_vs_compleja.csv` |
| Análisis escrito | sección 6 |

**Para reproducir:** subir este notebook y `corpus_metamorfosis.txt` a Google Colab, ejecutar todas las celdas.